In [ ]:
#Set up paths
import sys
import os

from Utils import utils as uti


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Some other useful packages 
import importlib
import copy
import time
import cftime
import yaml
import glob
#from box import Box #???
from pathlib import Path



In [ ]:
year,month,day,hour=2016,9,10,12

f3=f'/glade/campaign/cgd/amp/juliob/mpasa3p75km/TimeInvariant/zgrid_dyamond_fv1x1.nc' 
Xzg=xr.open_dataset( f3 )

for day in np.arange( start=1, stop=32 ):
    for hour in np.arange( 24, step=3 ):
        seconds=hour*3_600
        date_=f"{year:04d}-{month:02d}-{day:02d}-{seconds:05d}"          #{day:02d}-{hour*3_600:05d}"

        f1=f'/glade/campaign/cgd/amp/juliob/mpasa3p75km/{date_}/PINT_dyamond_fv1x1.{date_}.nc'
        f2=f'/glade/derecho/scratch/juliob/archive/c124_dyamond1/atm/hist/DynVars_dyamond_fv1x1.{date_}.nc' 

        if Path(f1).exists():
            #print( f'{f1} exists' )
            Xpi=xr.open_dataset( f1 )
            Xdy=xr.open_dataset( f2 )
            
            Xdy['PINT'] = Xpi['PINT']
            Xdy['zgrid'] = Xzg['zgrid']
            
            # load into memory & close file before overwrite
            Xdy = Xdy.load()
            Xdy.close()
        
            tmp = f2 + ".tmp"
            Xdy.to_netcdf(tmp)
            os.replace(tmp, f2)
            print( f"Placed PINT and zgrid in {f2} ")


In [ ]:
Xzg

In [ ]:
Xdy

In [ ]:
import glob
import os

pattern = "
files = sorted(glob.glob(pattern))

print(f"Found {len(files)} files")

In [ ]:
for f in files:
    print(f"Processing: {f}")

    ds = xr.open_dataset(f)

    if "PS" in ds.variables:
        print("  PS already present — skipping")
        ds.close()
        continue

    if "PINT" not in ds.variables:
        print("  WARNING: no PINT variable — skipping")
        ds.close()
        continue

    # create PS
    ds["PS"] = ds["PINT"].isel(ilev=93)

    # load into memory & close file before overwrite
    ds = ds.load()
    ds.close()

    tmp = f + ".tmp"
    ds.to_netcdf(tmp)
    os.replace(tmp, f)

    print("  PS added")

print("Done.")